# Interpretable mammography lesion classifier (concept bottleneck) on CBIS-DDSM
**Research prototype. Not a medical device.**

**Design decisions**
- **Data:** CBIS-DDSM (open, biopsy-proven labels, lesion masks, radiologist concept annotations). Official test split is touched once. External validation on a second dataset (INbreast or VinDr-Mammo) is still required and is not in this notebook.
- **Task:** lesion-level benign vs malignant from a region of interest (mammography needs a localised region; whole-image screening is a different, harder problem).
- **No black box at the decision step:** a CNN predicts radiologists' own concepts (lesion type, mass shape, margins, calcification type and distribution, breast density). A logistic regression on those concept probabilities makes the decision, so every coefficient and contribution is readable and a clinician can correct a concept. The CNN that *perceives* the concepts is still a neural network, so we check its concept accuracy and show Grad-CAM maps. BI-RADS assessment and subtlety are excluded on purpose: they are downstream judgements that leak the label.
- **Safety layers:** patient-level splits, calibration, class-conditional conformal prediction (an "uncertain, refer" zone with a stated error rate), and an out-of-distribution flag.

**Before running:** edit `GITHUB_REPO` in the setup cell, download CBIS-DDSM from TCIA (the four case-description CSVs plus images). Copy images to the Colab local disk first if Drive is slow. A GPU runtime is needed.

In [ ]:
REPO = 'https://github.com/cyprio101/Breast_cancer_screening'
!rm -rf /content/Breast_cancer_screening
!git clone -q $REPO /content/Breast_cancer_screening
%pip install -q -e /content/Breast_cancer_screening
# Fallback so the import works even before the runtime restarts
import sys; sys.path.insert(0, '/content/Breast_cancer_screening/src')
import mammo_cbm; print('mammo_cbm', mammo_cbm.__version__, 'imported from', mammo_cbm.__file__)

In [1]:
from google.colab import drive; drive.mount('/content/drive')
import re, json, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
WORK = Path('/content/drive/MyDrive/Breast_cancer_screening'); WORK.mkdir(exist_ok=True)
CBIS_ROOT = Path('/content/drive/MyDrive/CBIS-DDSM')   # EDIT: folder with the 4 CSVs and the image folders
import mammo_cbm as M
SEED, ALPHA, EPOCHS = 42, 0.10, 25                      # ALPHA = per-class error level for the conformal sets
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED); np.random.seed(SEED); print(DEVICE)

Mounted at /content/drive


ModuleNotFoundError: No module named 'mammo_cbm'

## 1. Build the lesion table and concept vocabularies

In [ ]:
def read(name, lesion):
    d = pd.read_csv(CBIS_ROOT / name); d.columns = [c.strip().lower().replace(' ', '_') for c in d.columns]
    d['lesion'] = lesion; return d
tr = pd.concat([read('mass_case_description_train_set.csv', 'mass'), read('calc_case_description_train_set.csv', 'calc')], ignore_index=True)
te = pd.concat([read('mass_case_description_test_set.csv', 'mass'), read('calc_case_description_test_set.csv', 'calc')], ignore_index=True)

SRC = {'lesion': 'lesion', 'shape': 'mass_shape', 'margins': 'mass_margins', 'calc_type': 'calc_type',
       'calc_dist': 'calc_distribution', 'density': 'breast_density'}
def clean(s): return s.fillna('NONE').astype(str).str.strip().str.upper().replace({'': 'NONE', 'NAN': 'NONE'})
for d in (tr, te):
    for h, c in SRC.items(): d[h] = clean(d[c])
    d['y'] = (d['pathology'].str.upper() == 'MALIGNANT').astype(int)

HEADS = {}                      # classes with >= 30 training lesions; the rest become OTHER
for h in SRC:
    vc = tr[h].value_counts(); keep = sorted(vc[vc >= 30].index)
    HEADS[h] = keep + (['OTHER'] if (vc < 30).any() else [])
def enc(d): return np.stack([d[h].map(lambda v, h=h: HEADS[h].index(v) if v in HEADS[h] else HEADS[h].index('OTHER')) for h in HEADS], 1)
tr_Y, te_Y = enc(tr), enc(te)
print({h: len(v) for h, v in HEADS.items()}, '| malignant fraction:', round(tr.y.mean(), 3), '| train/test lesions:', len(tr), len(te))

## 2. Locate images and cache one lesion crop each
Crops are taken from the full mammogram around the ROI mask, with 25% context. The CBIS-DDSM CSVs are known to swap the crop and mask paths in some rows, so the mask is chosen as the ROI file whose shape equals the full image. **The folder-indexing cell assumes the TCIA layout and was written without access to your files; adapt it if you use a Kaggle JPEG version.** Failed rows are counted and dropped.

In [ ]:
pat = re.compile(r'(Mass|Calc)-(Training|Test)_P_\d+_(LEFT|RIGHT)_(CC|MLO)(_\d+)?')
index = {}
for f in CBIS_ROOT.rglob('*'):
    if f.suffix.lower() in ('.dcm', '.jpg', '.jpeg', '.png'):
        hit = next((x for x in f.parts if pat.fullmatch(x)), None)
        if hit: index.setdefault(hit, []).append(f)
key = lambda p: str(p).strip().split('/')[0]
print('indexed folders:', len(index))

def lesion_gray(r):
    try:
        full = M.load_array(index[key(r.image_file_path)][0])
        cands = [M.load_array(f) for k in {key(r.roi_mask_file_path), key(r.cropped_image_file_path)} for f in index.get(k, [])]
        mask = next(m for m in cands if m.shape == full.shape)
        ys, xs = np.nonzero(mask > 0.5 * mask.max())
        return (M.prep_gray(M.roi_crop(full, (ys.min(), xs.min(), ys.max() + 1, xs.max() + 1))) * 255).astype(np.uint8)
    except Exception: return None

def cache(d, Y):
    g = [lesion_gray(r) for r in d.itertuples()]; ok = np.array([x is not None for x in g])
    print(f'usable lesions: {ok.sum()}/{len(ok)}')
    return d[ok].reset_index(drop=True), Y[ok], np.stack([x for x in g if x is not None])
tr, tr_Y, tr_G = cache(tr, tr_Y); te, te_Y, te_G = cache(te, te_Y)

## 3. Patient-level split: train 70% / validation 15% (early stopping, decision layer) / calibration 15% (conformal, OOD)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
g = tr.patient_id.values
a, rest = next(GroupShuffleSplit(1, test_size=0.30, random_state=SEED).split(tr, groups=g))
v, c = next(GroupShuffleSplit(1, test_size=0.50, random_state=SEED).split(rest, groups=g[rest])); v, c = rest[v], rest[c]
assert not (set(g[a]) & set(g[v])) and not (set(g[a]) & set(g[c])) and not (set(g[v]) & set(g[c]))
print('train/val/cal/test:', len(a), len(v), len(c), len(te))

## 4. Train the concept network

In [ ]:
import torchvision.transforms.v2 as T
from torch.utils.data import Dataset, DataLoader
aug = T.Compose([T.RandomHorizontalFlip(), T.RandomAffine(10, translate=(.05, .05), scale=(.9, 1.1))])
class DS(Dataset):
    def __init__(s, G, Y, train): s.G, s.Y, s.t = G, torch.as_tensor(Y), train
    def __len__(s): return len(s.G)
    def __getitem__(s, i):
        g = torch.from_numpy(s.G[i].astype(np.float32) / 255.)[None]
        if s.t: g = aug(g)
        return M.normalise(g[0]), s.Y[i]
dl = lambda ix, train: DataLoader(DS(tr_G[ix], tr_Y[ix], train), batch_size=32, shuffle=train, num_workers=2)
model = M.CBM(HEADS).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
def run(loader, train):
    model.train(train); tot = n = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE); out, _ = model(x)
            loss = sum(F.cross_entropy(o, y[:, j], label_smoothing=0.05) for j, o in enumerate(out.values()))
            if train: opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(x); n += len(x)
    return tot / n
best, wait = 1e9, 0
for ep in range(EPOCHS):
    trl, vl = run(dl(a, True), True), run(dl(v, False), False); sched.step()
    print(f'epoch {ep+1:2d}  train {trl:.3f}  val {vl:.3f}')
    if vl < best: best, wait = vl, 0; torch.save(model.state_dict(), WORK / 'best.pt')
    else:
        wait += 1
        if wait >= 5: break
model.load_state_dict(torch.load(WORK / 'best.pt')); model.eval()

## 5. Concept probabilities, then the transparent decision layer

In [ ]:
@torch.no_grad()
def predict(G):
    P, Fe = [], []
    for i in range(0, len(G), 64):
        x = torch.stack([M.normalise(torch.from_numpy(z.astype(np.float32) / 255.)) for z in G[i:i + 64]]).to(DEVICE)
        out, f = model(x); P.append(torch.cat([F.softmax(o, 1) for o in out.values()], 1).cpu().numpy()); Fe.append(f.cpu().numpy())
    return np.concatenate(P), np.concatenate(Fe)
Pa, Fa = predict(tr_G[a]); Pv, _ = predict(tr_G[v]); Pc, Fc = predict(tr_G[c]); Pt, _ = predict(te_G)
yv, yc, yt = tr.y.values[v], tr.y.values[c], te.y.values

from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GroupKFold
folds = list(GroupKFold(5).split(Pv, yv, g[v]))
clf = LogisticRegressionCV(Cs=np.logspace(-2, 2, 9), cv=folds, scoring='neg_log_loss', max_iter=3000).fit(Pv, yv)
names = [f'{h}: {k}' for h, cl in HEADS.items() for k in cl]
coef_table = pd.Series(clf.coef_[0], index=names).sort_values(key=np.abs, ascending=False)
display(coef_table.head(15).to_frame('log-odds coefficient'))

## 6. Evaluate once on the official test set
Reports AUC with a patient-cluster bootstrap interval, calibration, concept accuracy, and an ablation using *true* concepts (the ceiling if the CNN saw concepts perfectly). The gap between the two rows is the price of the CNN's perception errors, not of interpretability.

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
pt = clf.predict_proba(Pt)[:, 1]; gt = te.patient_id.values
def auc_ci(y, s, grp, n=1000):
    rng = np.random.default_rng(SEED); ids = np.unique(grp); by = {k: np.where(grp == k)[0] for k in ids}; out = []
    for _ in range(n):
        ix = np.concatenate([by[k] for k in rng.choice(ids, len(ids))])
        if len(set(y[ix])) == 2: out.append(roc_auc_score(y[ix], s[ix]))
    return np.percentile(out, [2.5, 97.5])
lo, hi = auc_ci(yt, pt, gt)
print(f'Test AUC {roc_auc_score(yt, pt):.3f} (95% CI {lo:.3f} to {hi:.3f}) | Brier {brier_score_loss(yt, pt):.3f} '
      f'(prevalence-only {brier_score_loss(yt, np.full(len(yt), yv.mean())):.3f})')
onehot = lambda Y: np.concatenate([np.eye(len(cl))[Y[:, j]] for j, cl in enumerate(HEADS.values())], 1)
oracle = LogisticRegressionCV(Cs=np.logspace(-2, 2, 9), cv=folds, scoring='neg_log_loss', max_iter=3000).fit(onehot(tr_Y[v]), yv)
print(f'Ablation, true concepts: test AUC {roc_auc_score(yt, oracle.predict_proba(onehot(te_Y))[:, 1]):.3f}')
s = 0
for j, (h, cl) in enumerate(HEADS.items()):
    print(f'concept accuracy {h:10s}: {(Pt[:, s:s + len(cl)].argmax(1) == te_Y[:, j]).mean():.3f}'); s += len(cl)
fp, mp = calibration_curve(yt, pt, n_bins=5, strategy='quantile')
plt.plot([0, 1], [0, 1], 'k--'); plt.plot(mp, fp, 'o-'); plt.xlabel('predicted'); plt.ylabel('observed malignant'); plt.title('Test calibration'); plt.show()

## 7. Uncertainty: class-conditional conformal sets and the out-of-distribution flag
For each class the threshold is set on the calibration split so that about 90% of true cases of that class keep their correct label in the prediction set. A set holding both labels becomes **Uncertain, refer**. The OOD flag marks images far from the training feature distribution (set at the 99th percentile of calibration images).

In [ ]:
pc = clf.predict_proba(Pc)[:, 1]
q = M.conformal_thresholds(pc, yc, ALPHA)
verd = np.array([M.verdict_from_p(p, q) for p in pt])
for lab in ('Suspicious', 'Likely benign', 'Uncertain'): print(f'{lab:14s} n={np.sum(verd == lab):4d}  malignant fraction {yt[verd == lab].mean() if np.any(verd == lab) else float("nan"):.2f}')
mal = yt == 1; ben = ~mal
print(f'Malignant lesions not labelled Likely benign: {np.mean(verd[mal] != "Likely benign"):.3f}')
print(f'Benign lesions not labelled Suspicious     : {np.mean(verd[ben] != "Suspicious"):.3f}')
print(f'Abstain rate (Uncertain): {np.mean(verd == "Uncertain"):.3f}')

from sklearn.covariance import LedoitWolf
lw = LedoitWolf().fit(Fa); dc = lambda X: np.einsum('ij,jk,ik->i', X - lw.location_, lw.precision_, X - lw.location_)
ood_thr = np.percentile(dc(Fc), 99)

## 8. Save the bundle for the dashboard

In [ ]:
meta = dict(dataset='CBIS-DDSM', lesions_train=int(len(a)), test_auc=float(roc_auc_score(yt, pt)), test_auc_ci=[float(lo), float(hi)],
            conformal_alpha=ALPHA, intended_use='research prototype, ROI-based second reader, not for clinical use',
            external_validation='NOT DONE')
M.save_bundle(WORK / 'mammo_cbm_bundle.pt', model.cpu(), HEADS, 'efficientnet_b0', clf.coef_[0], clf.intercept_[0],
              Pa.mean(0), q, lw.location_, lw.precision_, ood_thr, meta)
json.dump(meta, open(WORK / 'model_card.json', 'w'), indent=2); print('saved to', WORK)

## What is still missing before this could be used on patients
1. **External validation** on another site, scanner and population (INbreast or VinDr-Mammo), with the frozen bundle and untouched thresholds.
2. **Subgroup performance** by breast density, lesion type and age where available.
3. **Reader study:** does the tool improve radiologists' accuracy, and do they trust the explanations appropriately?
4. **Localisation check:** compare Grad-CAM with the ROI masks (pointing game).
5. **Regulatory and ethics approval**, a detector for whole-image use, and post-deployment monitoring.